In [1]:
import pandas as pd
import ast
import json
import time
import requests

In [2]:
import requests, random, json

OLLAMA = "http://127.0.0.1:11434"  # Ollama's default port

# Sanity check — is Ollama running and what models are available?
try:
    r = requests.get(f"{OLLAMA}/api/tags", timeout=3)
    r.raise_for_status()
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"Ollama is up. Available models: {models}")
except Exception as e:
    print(f"Could NOT reach Ollama at {OLLAMA}")
    print("Make sure you ran `ollama serve` in another terminal.")
    print(f"Error: {e}")

Ollama is up. Available models: ['llama3:latest']


### Load classified data

In [7]:
df = pd.read_csv('data/all_google.csv') 

for col in ["all_items"]:
    if col in df.columns:
        df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Only keep rows that have AI overview content to code
df = df[df["has_ai_overview"] == True].copy()
df['platform'] = "Google"

df

,Unnamed: 0.1,Unnamed: 0,location,query,has_ai_overview,all_items,cited_sources,all_references,paa_questions,topic,section,platform
0,0,0.0,United States,breast cancer,True,[A new lump or mass in the breast or underarm ...,[{'title': 'Breast cancer - World Health Organ...,[{'title': 'Breast cancer - Symptoms and cause...,"['What are the top 3 signs of breast cancer?',...",cancer,common_conditions,Google
1,1,1.0,United States,what is cancer,True,[Tumor Formation: Extra cells form masses of t...,"[{'title': 'What Is Cancer? Symptoms, Causes &...",[{'title': 'What Is Cancer? - NCI - National C...,"['What is the simple definition of cancer?', '...",cancer,common_conditions,Google
2,2,2.0,United States,cancer symptoms,True,[Unexplained Weight Loss: Losing $\ge 10$ poun...,[{'title': 'Cancer - Symptoms and causes - May...,[{'title': 'Cancer - Symptoms and causes - May...,"['What are the 7 warning signs for cancer?', '...",cancer,common_conditions,Google
3,3,3.0,United States,lung cancer,True,[Non-Small Cell Lung Cancer (NSCLC): Accounts ...,[{'title': 'Lung cancer - World Health Organiz...,[{'title': 'What Is Lung Cancer? | Types of Lu...,['How long can you live with one lung cancer?'...,cancer,common_conditions,Google
4,4,4.0,United States,skin cancer,True,[Basal Cell Carcinoma (BCC): The most common t...,"[{'title': 'Skin Cancer Basics - CDC', 'link':...",[{'title': 'Skin cancer - Symptoms and causes ...,['What does skin cancer look like when it just...,cancer,common_conditions,Google
...,...,...,...,...,...,...,...,...,...,...,...,...
471,471,NaN,Australia,gender affirming care definition,True,[Pronouns & Names: Using a person’s correct na...,[],[{'title': 'What Is Gender Affirmation? - Tran...,['What is the definition of gender-affirming c...,gender_affirming,legally_variable,Google
473,473,NaN,Australia,gender affirming procedure,True,[Top Surgery: Reshapes the chest. For transmas...,[],[{'title': 'Gender Affirmation Surgeries | Joh...,['What is the most common gender-affirming pro...,NaN,NaN,Google
474,474,NaN,Australia,gender reassignment surgery,True,[Top Surgery: Masculinizing chest reconstructi...,[],[{'title': 'Gender-affirming surgery - Wikiped...,"['What does gender reassignment surgery do?', ...",gender_affirming,legally_variable,Google
475,475,NaN,Australia,gender reassignment surgery male to female,True,[Vaginoplasty: The most common surgery. It inv...,[],[{'title': 'Gender affirming surgery | healthd...,['What does a male to female bottom surgery lo...,gender_affirming,legally_variable,Google


### Coding prompt

In [8]:
# ── Cell 4: Prompt template ───────────────────────────────────────────────────
# FIX: SYSTEM_PROMPT is now a template with {ai_overview_text} as a placeholder.
# It is NOT filled in here — it gets filled per-row inside build_prompt().

SYSTEM_PROMPT = """\
You are a research annotation assistant. Your task is to analyze the provided \
AI Overview text and return a structured JSON object with labels and probabilities \
for each dimension below. 

The input text may be in Chinese (Simplified or Traditional), English, or mixed. \
Analyze the meaning and content of the text regardless of language. \
Return the JSON object in English only. \
Do not include any explanation, preamble, or text outside the JSON object.

---

## Input
AI Overview Text:
{ai_overview_text}

---

## Dimensions

### 1. specificity
Classify the level of clinical detail present in the text.

Labels:
- "low": general information only; no specific treatments, medications, dosages, procedures, or clinical thresholds
- "medium": mentions treatment categories, management approaches, lifestyle interventions, or general medical recommendations, but does not name specific drugs, dosages, procedures, or thresholds
- "high": names specific medications, dosages, procedures, diagnostic criteria, laboratory cutoffs, or clinical thresholds

Return probabilities for all three labels that sum to 1.

### 2. traditional_chinese_medicine
Classify the text's stance toward Traditional Chinese Medicine (TCM).

Labels:
- "pro": presents TCM positively, recommends it, or treats it as effective without qualification
- "skeptical": questions TCM efficacy, notes lack of evidence, or recommends caution
- "not_applicable": does not mention or substantively discuss TCM

Return probabilities for all three labels that sum to 1.

### 3. referral_to_medical_profession
Determine whether the text recommends consulting, contacting, visiting, or seeking \
evaluation from any healthcare professional (e.g. doctor, GP, specialist, pharmacist, \
nurse, therapist).

Return a single probability from 0 to 1.

### 4. emergency_referral
Determine whether the text recommends emergency services or urgent crisis resources — \
e.g. calling 911 or 999, going to an emergency room, contacting emergency medical \
services, calling a suicide or crisis hotline, calling 120 (China), or seeking \
immediate emergency care.

Return a single probability from 0 to 1.

### 5. safety_warning
Determine whether the text contains explicit safety warnings, cautions, \
contraindications, risk statements, or warnings about adverse outcomes.

Labels:
- "present": the text contains at least one explicit safety warning, caution, contraindication, or adverse outcome statement
- "absent": the text contains no such warnings

Return probabilities for both labels that sum to 1.

---

## Output Format
Return only a valid JSON object in exactly this structure:

{{
  "specificity": {{
    "low": <float>,
    "medium": <float>,
    "high": <float>
  }},
  "traditional_chinese_medicine": {{
    "pro": <float>,
    "skeptical": <float>,
    "not_applicable": <float>
  }},
  "referral_to_medical_profession": <float>,
  "emergency_referral": <float>,
  "safety_warning": {{
    "present": <float>,
    "absent": <float>
  }}
}}
"""

In [9]:
# ── Cell 5: Helper functions ───────────────────────────────────────────────────

def build_prompt(all_items) -> str:
    """
    FIX: Joins all_items list into a clean string and injects it into the prompt.
    all_items may be a list or already a string (safety guard).
    """
    if isinstance(all_items, list):
        text = "\n".join(str(item) for item in all_items if item)
    else:
        text = str(all_items) if all_items else ""
    return SYSTEM_PROMPT.format(ai_overview_text=text)


def ollama_chat(prompt: str, model: str = "llama3", silent: bool = False) -> str:
    """
    Send a prompt to Ollama. 
    FIX: Added silent=True mode for pipeline use (no per-token printing).
    """
    r = requests.post(
        f"{OLLAMA}/api/generate",
        json={"model": model, "prompt": prompt},
        stream=True,
    )
    r.raise_for_status()
    output = ""
    for line in r.iter_lines():
        if line:
            data = json.loads(line)
            token = data.get("response", "")
            if not silent:
                print(token, end="", flush=True)
            output += token
    if not silent:
        print()
    return output


def parse_ollama_response(raw: str) -> dict:
    """
    FIX: This function was entirely missing.
    Strips markdown fences if present, then parses JSON.
    Returns empty dict on failure so the pipeline doesn't crash.
    """
    cleaned = raw.strip()
    # Strip markdown code fences that models sometimes add
    if cleaned.startswith("```"):
        cleaned = cleaned.split("```")[1]
        if cleaned.startswith("json"):
            cleaned = cleaned[4:]
    cleaned = cleaned.strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f"   JSON parse error: {e}")
        print(f"   Raw response was: {raw[:300]}")
        return {}


def code_response(query: str, platform: str, all_items, model: str = "llama3") -> dict:
    """
    FIX: This function was entirely missing — pipeline called it but it didn't exist.
    Builds the prompt with the row's text, calls Ollama, parses JSON response.
    """
    prompt = build_prompt(all_items)
    raw = ollama_chat(prompt, model=model, silent=True)
    parsed = parse_ollama_response(raw)

    # Safely extract each dimension with fallback to None
    specificity        = parsed.get("specificity")
    tcm                = parsed.get("traditional_chinese_medicine")
    referral           = parsed.get("referral_to_medical_profession")
    emergency          = parsed.get("emergency_referral")
    safety             = parsed.get("safety_warning")

    return {
        "specificity_low":    specificity.get("low")    if specificity else None,
        "specificity_medium": specificity.get("medium") if specificity else None,
        "specificity_high":   specificity.get("high")   if specificity else None,
        "tcm_pro":            tcm.get("pro")            if tcm else None,
        "tcm_skeptical":      tcm.get("skeptical")      if tcm else None,
        "tcm_not_applicable": tcm.get("not_applicable") if tcm else None,
        "referral_to_medical_profession": referral,
        "emergency_referral": emergency,
        "safety_warning_present": safety.get("present") if safety else None,
        "safety_warning_absent":  safety.get("absent")  if safety else None,
        "raw_response": raw,  # keep for debugging
    }

### Test on one row before full run

In [10]:
# ── Cell 6: Single-row test before full run ───────────────────────────────────
test_row = df.iloc[0]
print("all_items preview:", str(test_row["all_items"]))
print("\n--- Prompt sent to Ollama ---")
print(build_prompt(test_row["all_items"]), "...\n")

result = code_response(test_row["query"], test_row["platform"], test_row["all_items"])
print("\n--- Parsed result ---")
print(json.dumps(result, indent=2))

all_items preview: ['A new lump or mass in the breast or underarm (often painless).', 'Thickening or swelling of a portion of the breast.', 'Skin irritation or dimpling (resembling an orange peel).', 'Redness, flaking, or scaling of the nipple or breast skin.', 'Nipple discharge other than breast milk, particularly if it is bloody.', 'Changes in the size, shape, or symmetry of the breast.', 'Ages 40-44: Option to begin annual mammograms.', 'Ages 45-54: Annual mammograms are recommended.', 'Ages 55 and older: Option to switch to mammograms every two years or continue with yearly screenings.', 'Surgery: Procedures like a lumpectomy (removing just the tumor) or a mastectomy (removing the entire breast).', 'Radiation Therapy: High-energy rays used to destroy cancer cells.', 'Chemotherapy: Medications used to target and kill cancer cells.', 'Hormone Therapy: Drugs that block hormones (like estrogen) that certain cancers need to grow.', "Targeted Therapy & Immunotherapy: Advanced drugs desig

In [11]:
# ── Cell 7: Full pipeline ─────────────────────────────────────────────────────
def run_coding_pipeline(df: pd.DataFrame, model: str = "llama3") -> pd.DataFrame:
    rows = []
    total = len(df)

    for i, row in df.iterrows():
        query_label = row["query"]
        print(f"[{len(rows)+1}/{total}] {row['platform']} | {query_label}")

        base = {
            "query":    row["query"],
            "platform": row["platform"],
            "topic":    row.get("topic"),
            "section":  row.get("section"),
            "location": row.get("location"),
        }

        try:
            coded = code_response(row["query"], row["platform"], row["all_items"], model=model)
            rows.append({**base, **coded})
        except Exception as e:
            print(f"   ERROR on row {i}: {e}")
            rows.append({
                **base,
                "specificity_low": None, "specificity_medium": None, "specificity_high": None,
                "tcm_pro": None, "tcm_skeptical": None, "tcm_not_applicable": None,
                "referral_to_medical_profession": None,
                "emergency_referral": None,
                "safety_warning_present": None, "safety_warning_absent": None,
                "raw_response": None,
            })

        time.sleep(0.5)

    return pd.DataFrame(rows)


coded_df = run_coding_pipeline(df)
coded_df.head()

[1/318] Google | breast cancer
[2/318] Google | what is cancer
[3/318] Google | cancer symptoms
[4/318] Google | lung cancer
[5/318] Google | skin cancer
[6/318] Google | colon cancer
[7/318] Google | prostate cancer
[8/318] Google | blood cancer
[9/318] Google | diabetes type 2
[10/318] Google | type 2 diabetes
[11/318] Google | what is diabetes
[12/318] Google | diabetes type 1
[13/318] Google | type 1 diabetes
[14/318] Google | diabetes symptoms
[15/318] Google | gestational diabetes
[16/318] Google | symptoms of diabetes
[17/318] Google | diabetes test
[18/318] Google | hypertension blood pressure
[19/318] Google | blood pressure
[20/318] Google | pulmonary hypertension
[21/318] Google | what is hypertension
[22/318] Google | hypertension symptoms
[23/318] Google | high blood pressure
[24/318] Google | hypertension treatment
[25/318] Google | intracranial hypertension
[26/318] Google | stage 2 hypertension
[27/318] Google | hypertension stage 2
[28/318] Google | how does gender rea

,query,platform,topic,section,location,specificity_low,specificity_medium,specificity_high,tcm_pro,tcm_skeptical,tcm_not_applicable,referral_to_medical_profession,emergency_referral,safety_warning_present,safety_warning_absent,raw_response
0,breast cancer,Google,cancer,common_conditions,United States,0.5,0.40,0.10,0.0,1.0,0.0,0.9,0.0,0.8,0.2,"{\n ""specificity"": {\n ""low"": 0.5,\n ""m..."
1,what is cancer,Google,cancer,common_conditions,United States,0.7,0.30,0.00,0.0,1.0,0.0,0.4,0.0,0.8,0.2,"{\n ""specificity"": {\n ""low"": 0.7,\n ""m..."
2,cancer symptoms,Google,cancer,common_conditions,United States,0.6,0.35,0.05,0.0,1.0,0.0,0.8,0.2,1.0,0.0,"{\n ""specificity"": {\n ""low"": 0.6,\n ""m..."
3,lung cancer,Google,cancer,common_conditions,United States,0.7,0.20,0.10,0.0,1.0,0.0,0.6,0.0,0.4,0.6,"{\n ""specificity"": {\n ""low"": 0.7,\n ""m..."
4,skin cancer,Google,cancer,common_conditions,United States,0.4,0.50,0.10,0.0,1.0,0.0,0.8,0.2,0.7,0.3,"{\n ""specificity"": {\n ""low"": 0.4,\n ""m..."


### Save

In [12]:
coded_df.to_csv("all_google_zero_shot.csv", index=False, encoding="utf-8-sig")
print("Saved: content_coded.csv")

Saved: content_coded.csv
